# 可选实验：特征工程和多项式回归

![](./images/C1_W2_Lab07_FeatureEngLecture.png)


## 目标
在这个实验中，你将：
- 探索特征工程和多项式回归，这允许你使用线性回归的机制来拟合非常复杂，甚至是非常非线性的函数。


## 工具
你将使用以前实验中开发的函数以及 matplotlib 和 NumPy。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import zscore_normalize_features, run_gradient_descent_feng
np.set_printoptions(precision=2)  # reduced display precision on numpy arrays

<a name='FeatureEng'></a>
# 特征工程和多项式回归概述

开箱即用，线性回归提供了一种构建以下形式模型的方法：
$$f_{\mathbf{w},b} = w_0x_0 + w_1x_1+ ... + w_{n-1}x_{n-1} + b \tag{1}$$ 
如果你的特征/数据是非线性的或者是特征的组合呢？例如，房价往往不随居住面积线性变化，而是对非常小或非常大的房子进行惩罚，导致上图所示的曲线。我们如何使用线性回归的机制来拟合这条曲线？回想一下，我们拥有的“机制”是修改 (1) 中参数 $\mathbf{w}$, $\mathbf{b}$ 以将方程“拟合”到训练数据的能力。然而，无论如何调整 (1) 中的 $\mathbf{w}$,$\mathbf{b}$ 都无法实现对非线性曲线的拟合。


<a name='PolynomialFeatures'></a>
## 多项式特征

在上面我们考虑了数据是非线性的场景。让我们尝试使用我们要知道的方法来拟合非线性曲线。我们将从一个简单的二次方程开始：$y = 1+x^2$

你熟悉我们使用的所有例程。它们在 lab_utils.py 文件中可供查看。我们将使用 [`np.c_[..]`](https://numpy.org/doc/stable/reference/generated/numpy.c_.html)，这是一个 NumPy 例程，用于沿列边界连接。

In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = 1 + x**2
X = x.reshape(-1, 1)

model_w,model_b = run_gradient_descent_feng(X,y,iterations=1000, alpha = 1e-2)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("no feature engineering")
plt.plot(x,X@model_w + model_b, label="Predicted Value");  plt.xlabel("X"); plt.ylabel("y"); plt.legend(); plt.show()

嗯，正如预期的那样，拟合得不是很好。需要的是像 $y= w_0x_0^2 + b$ 这样的东西，或者是一个 **多项式特征**。
为了实现这一点，你可以修改 *输入数据* 来 *设计* 所需的特征。如果你将原始数据与平方 $x$ 值的版本交换，那么你可以实现 $y= w_0x_0^2 + b$。让我们试试。在下面将 `X` 换成 `X**2`：

In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = 1 + x**2

# Engineer features 
X = x**2      #<-- added engineered feature

In [ ]:
X = X.reshape(-1, 1)  #X should be a 2-D Matrix
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha = 1e-5)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Added x**2 feature")
plt.plot(x, np.dot(X,model_w) + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

太棒了！近乎完美的拟合。注意图表正上方打印的 $\mathbf{w}$ 和 b 的值：`w,b found by gradient descent: w: [1.], b: 0.0490`。梯度下降将我们的 $\mathbf{w},b$ 初始值修改为 (1.0,0.049) 或模型 $y=1*x_0^2+0.049$，非常接近我们的目标 $y=1*x_0^2+1$。如果你运行更长时间，它可能会更好地匹配。

### 选择特征
<a name='GDF'></a>
在上面，我们知道需要一个 $x^2$ 项。哪些特征是必需的并不总是显而易见的。可以添加各种潜在特征来尝试找到最有用的特征。例如，如果我们尝试了：$y=w_0x_0 + w_1x_1^2 + w_2x_2^3+b$ 呢？

运行下一个单元格。

In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = x**2

# engineer features .
X = np.c_[x, x**2, x**3]   #<-- added engineered feature

In [ ]:
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha=1e-7)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("x, x**2, x**3 features")
plt.plot(x, X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

注意 $\mathbf{w}$ 的值 `[0.08 0.54 0.03]` 和 b 是 `0.0106`。这意味着拟合/训练后的模型是：
$$ 0.08x + 0.54x^2 + 0.03x^3 + 0.0106 $$
梯度下降通过增加相对于其他项的 $w_1$ 项，强调了最适合 $x^2$ 数据的数据。如果你运行很长时间，它将继续减少其他项的影响。
>梯度下降通过强调其关联参数为我们挑选“正确”的特征

让我们回顾一下这个想法：
- 最初，特征被重新缩放，以便它们彼此可比
- 较小的权重值意味着较不重要/正确的特征，在极端情况下，当权重变为零或非常接近零时，相关特征在将模型拟合到数据中很有用。
- 在上面，拟合后，与 $x^2$ 特征关联的权重远大于 $x$ 或 $x^3$ 的权重，因为它是拟合数据最有用的。

### 另一种观点
在上面，多项式特征是根据它们与目标数据的匹配程度来选择的。另一种思考方式是注意一旦我们创建了新特征，我们仍然在使用线性回归。鉴于此，最好的特征将是相对于目标线性的。这最好通过示例来理解。

In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = x**2

# engineer features .
X = np.c_[x, x**2, x**3]   #<-- added engineered feature
X_features = ['x','x^2','x^3']

In [ ]:
fig,ax=plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X[:,i],y)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("y")
plt.show()

在上面，很明显 $x^2$ 特征映射到目标值 $y$ 是线性的。然后线性回归可以轻松地使用该特征生成模型。

### 缩放特征
正如在上一个实验中所述，如果数据集具有显著不同比例的特征，则应应用特征缩放以加速梯度下降。在上面的例子中，有 $x$, $x^2$ 和 $x^3$，它们自然会有非常不同的比例。让我们对我们的例子应用 Z-score 标准化。

In [ ]:
# create target data
x = np.arange(0,20,1)
X = np.c_[x, x**2, x**3]
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X,axis=0)}")

# add mean_normalization 
X = zscore_normalize_features(X)     
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X,axis=0)}")

现在我们可以用更激进的 alpha 值再试一次：

In [ ]:
x = np.arange(0,20,1)
y = x**2

X = np.c_[x, x**2, x**3]
X = zscore_normalize_features(X) 

model_w, model_b = run_gradient_descent_feng(X, y, iterations=100000, alpha=1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

特征缩放允许它收敛得快得多。
再次注意 $\mathbf{w}$ 的值。$w_1$ 项，即 $x^2$ 项，是最受强调的。梯度下降几乎消除了 $x^3$ 项。

### 复杂函数
通过特征工程，即使是相当复杂的函数也可以建模：

In [ ]:
x = np.arange(0,20,1)
y = np.cos(x/2)

X = np.c_[x, x**2, x**3,x**4, x**5, x**6, x**7, x**8, x**9, x**10, x**11, x**12, x**13]
X = zscore_normalize_features(X) 

model_w,model_b = run_gradient_descent_feng(X, y, iterations=1000000, alpha = 1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()



## 恭喜！
在这个实验中，你：
- 了解了线性回归如何使用特征工程对复杂甚至高度非线性的函数进行建模
- 认识到在进行特征工程时应用特征缩放很重要